# Intelli-Speech - RECOVERY

One-stop recovery notebook for a fresh or broken Colab runtime.

## Recovery order

| Step | Concern | Writes |
|---|---|---|
| 1 | Runtime diagnostics | none |
| 2 | Drive mount and repository sync | mounts Drive, clones the repo only if missing |
| 3 | Persistent cache configuration | creates cache directories in Drive |
| 4 | Hugging Face authentication state | none |
| 5 | Checkpoint recovery verification | none |
| 6 | ClearerVoice recovery | none |

Run the steps in order. Each concern appears exactly once; do not copy these cells into
`MAINFILE.ipynb`.

## Rules

- No package installation, upgrade, or downgrade happens in this notebook.
- No dataset download and no dataset regeneration happens in this notebook.
- No model is instantiated and no inference is run here.
- Installation and dependency changes require an explicit decision recorded in
  `PROJECT_NOTES.md`.


In [7]:
# ============================================================
# STEP 1 | RUNTIME DIAGNOSTICS (read-only)
# Mirrors scripts/setup_colab_runtime.py. Nothing is installed or changed.
# ============================================================
import importlib.metadata
import importlib.util
import platform
import sys
from pathlib import Path


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print("=" * 72)
print("INTELLI-SPEECH | RUNTIME DIAGNOSTICS")
print("=" * 72)
print(f"Python: {platform.python_version()} ({sys.executable})")
print(f"Platform: {platform.platform()}")
for package in (
    "numpy",
    "pandas",
    "torch",
    "torchaudio",
    "librosa",
    "soundfile",
    "huggingface-hub",
    "clearvoice",
):
    print(f"{package}: {package_version(package)}")

numpy_version = package_version("numpy")
if numpy_version != "not installed":
    major = numpy_version.split(".", 1)[0]
    if major.isdigit() and int(major) >= 2:
        print(
            "Note: ClearerVoice 0.1.2 declares numpy<2.0. The installed NumPy is "
            f"{numpy_version}. This mismatch is a known blocker; see PROJECT_NOTES.md."
        )

try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"PyTorch CUDA build: {torch.version.cuda}")
    if torch.cuda.is_available():
        print(f"CUDA device 0: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("CUDA available: unknown (torch not installed)")

in_colab = importlib.util.find_spec("google.colab") is not None
print(f"Google Colab environment: {in_colab}")
print(f"Drive root exists: {Path('/content/drive').is_dir()}")
print(f"Repository exists: {Path('/content/Intelli-Speech').is_dir()}")
print("No package installation, upgrade, or downgrade was performed.")
print("=" * 72)


INTELLI-SPEECH | RUNTIME DIAGNOSTICS
Python: 3.13.18 (/usr/bin/python3)
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
numpy: 2.5.5
pandas: 2.2.3
torch: 2.14.1+cpu
torchaudio: 2.11.0+cpu
librosa: 0.10.2.post1
soundfile: 0.12.1
huggingface-hub: 1.33.0
clearvoice: 0.1.2
Note: ClearerVoice 0.1.2 declares numpy<2.0. The installed NumPy is 2.5.5. This mismatch is a known blocker; see PROJECT_NOTES.md.
CUDA available: False
PyTorch CUDA build: None
Google Colab environment: True
Drive root exists: True
Repository exists: True
No package installation, upgrade, or downgrade was performed.


In [8]:
# ============================================================
# STEP 2 | DRIVE MOUNT AND REPOSITORY SYNC
# Mounts Drive when needed. Clones the repository only when it is missing.
# Never pulls, resets, commits, or pushes automatically.
# ============================================================
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
REPO_DIR = Path("/content/Intelli-Speech")
REPO_URL = "https://github.com/niharika9nx/Intelli-Speech.git"

if not DRIVE_ROOT.is_dir():
    from google.colab import drive

    drive.mount(str(DRIVE_ROOT))
print(f"Drive root exists: {DRIVE_ROOT.is_dir()}")

if not REPO_DIR.is_dir():
    print(f"Repository missing; cloning into {REPO_DIR}.")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Repository already present: {REPO_DIR}")

if REPO_DIR.is_dir():
    head = subprocess.run(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
        check=False,
        capture_output=True,
        text=True,
    )
    print("Current HEAD:", head.stdout.strip() or "unavailable")
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=False,
        capture_output=True,
        text=True,
    )
    dirty = status.stdout.strip()
    print("Working tree clean:", not dirty)
    if dirty:
        print("Local changes present; review them before pulling:")
        print(dirty)

print(
    "Repository sync is manual by design: run 'git pull origin main' yourself "
    "after reviewing the working tree."
)


Drive root exists: True
Repository already present: /content/Intelli-Speech
Current HEAD: e1b05f9 additions
Working tree clean: True
Repository sync is manual by design: run 'git pull origin main' yourself after reviewing the working tree.


In [9]:
# ============================================================
# STEP 3 | PERSISTENT CACHE CONFIGURATION
# Keeps pip, Hugging Face, and ClearerVoice caches on Drive across restarts.
# Creates directories only; downloads nothing.
# ============================================================
import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Intelli-Speech-data")
CACHE_ROOT = PROJECT_DIR / "persistent_cache"

CACHE_PATHS = {
    "pip": CACHE_ROOT / "pip",
    "huggingface": CACHE_ROOT / "huggingface",
    "huggingface_hub": CACHE_ROOT / "huggingface" / "hub",
    "clearvoice": CACHE_ROOT / "clearvoice_runtime",
}

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project Drive directory not found: {PROJECT_DIR}. "
        "Mount Drive and verify the dataset path before continuing."
    )

for path in CACHE_PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

os.environ["PIP_CACHE_DIR"] = str(CACHE_PATHS["pip"])
os.environ["HF_HOME"] = str(CACHE_PATHS["huggingface"])
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_PATHS["huggingface_hub"])

print("Persistent caches configured (no downloads performed):")
for name, path in CACHE_PATHS.items():
    files = [item for item in path.rglob("*") if item.is_file()]
    total_mb = sum(item.stat().st_size for item in files) / (1024 * 1024)
    print(f"  {name}: {path} | files={len(files)}, {total_mb:.2f} MB")


Persistent caches configured (no downloads performed):
  pip: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/pip | files=6, 0.19 MB
  huggingface: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/huggingface | files=0, 0.00 MB
  huggingface_hub: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/huggingface/hub | files=0, 0.00 MB
  clearvoice: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/clearvoice_runtime | files=0, 0.00 MB


In [10]:
# ============================================================
# STEP 4 | HUGGING FACE AUTHENTICATION STATE (read-only, offline)
# Reads the locally stored token only. No network call is made.
# Authenticate only if a gated asset actually requires it.
# ============================================================
import importlib.util
import os

print("=== HUGGING FACE AUTHENTICATION ===")

if importlib.util.find_spec("huggingface_hub") is None:
    print("huggingface_hub is not installed. No authentication is possible yet.")
else:
    from huggingface_hub import get_token

    token = get_token()
    print("Stored token present:", token is not None)
    print("HF_HOME:", os.environ.get("HF_HOME", "not set"))
    print(
        "HUGGINGFACE_HUB_CACHE:",
        os.environ.get("HUGGINGFACE_HUB_CACHE", "not set"),
    )
    if token is None:
        print(
            "No token stored. Only if a gated checkpoint requires authentication, "
            "run the login helper yourself in a separate cell:"
        )
        print("    from huggingface_hub import notebook_login")
        print("    notebook_login()")
        print(
            "Do not paste tokens into this notebook or into repository files."
        )
    else:
        print(
            "A token is already stored. Re-authenticate only after a runtime "
            "restart or an explicit authentication failure."
        )


=== HUGGING FACE AUTHENTICATION ===
Stored token present: False
HF_HOME: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/huggingface
HUGGINGFACE_HUB_CACHE: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/huggingface/hub
No token stored. Only if a gated checkpoint requires authentication, run the login helper yourself in a separate cell:
    from huggingface_hub import notebook_login
    notebook_login()
Do not paste tokens into this notebook or into repository files.


In [11]:
# ============================================================
# STEP 5 | CHECKPOINT RECOVERY VERIFICATION (read-only)
# Verifies the existing STFT analysis checkpoint and locates any FRCRN
# checkpoint. Nothing is downloaded, created, or overwritten.
# ============================================================
import importlib.util
from pathlib import Path

import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/Intelli-Speech-data")
DATASET_DIR = PROJECT_DIR / "paired_dataset_v3"
STFT_CHECKPOINT_PATH = PROJECT_DIR / "analysis_cache" / "stft_example.npz"

print("=== CHECKPOINT RECOVERY ===")
print("Dataset directory exists:", DATASET_DIR.is_dir())
print("Dataset metadata exists:", (DATASET_DIR / "metadata.csv").is_file())

if not STFT_CHECKPOINT_PATH.is_file():
    print(f"STFT checkpoint missing: {STFT_CHECKPOINT_PATH}")
    print("Do not regenerate it silently; report the missing artifact first.")
else:
    with np.load(STFT_CHECKPOINT_PATH) as saved:
        sample_rate = int(saved["sample_rate"])
        n_fft = int(saved["n_fft"])
        hop_length = int(saved["hop_length"])
        clean_stft = saved["clean_stft"]
        noisy_stft = saved["noisy_stft"]
        finite = bool(
            np.isfinite(clean_stft).all() and np.isfinite(noisy_stft).all()
        )
    print(f"STFT checkpoint: {STFT_CHECKPOINT_PATH}")
    print(f"  sample_rate={sample_rate} n_fft={n_fft} hop_length={hop_length}")
    print(f"  clean_stft={clean_stft.shape} noisy_stft={noisy_stft.shape}")
    print(f"  all values finite={finite}")

clearvoice_spec = importlib.util.find_spec("clearvoice")
checkpoint_dirs = {
    "CWD checkpoints/FRCRN_SE_16K": Path.cwd() / "checkpoints" / "FRCRN_SE_16K",
    "repo checkpoints/FRCRN_SE_16K": Path("/content/Intelli-Speech/checkpoints/FRCRN_SE_16K"),
    "/content checkpoints/FRCRN_SE_16K": Path("/content/checkpoints/FRCRN_SE_16K"),
    "persistent_cache/clearvoice_runtime": PROJECT_DIR / "persistent_cache" / "clearvoice_runtime",
    "persistent_cache/huggingface/hub": PROJECT_DIR / "persistent_cache" / "huggingface" / "hub",
    "user ClearerVoice cache": Path.home() / ".cache" / "clearvoice",
    "user Hugging Face cache": Path.home() / ".cache" / "huggingface" / "hub",
}
if clearvoice_spec is not None and clearvoice_spec.origin:
    checkpoint_dirs["clearvoice package checkpoints"] = (
        Path(clearvoice_spec.origin).parent / "checkpoints"
    )

checkpoint_suffixes = {
    ".pt", ".pth", ".ckpt", ".onnx", ".bin", ".safetensors", ".tar", ".gz", ".zip"
}
print("\nFRCRN checkpoint locations (read-only):")
found_any = False
for label, directory in checkpoint_dirs.items():
    if not directory.is_dir():
        print(f"  {label}: missing ({directory})")
        continue
    files = [
        item
        for item in directory.rglob("*")
        if item.is_file() and item.suffix.lower() in checkpoint_suffixes
    ]
    total_mb = sum(item.stat().st_size for item in files) / (1024 * 1024)
    print(f"  {label}: {directory}")
    print(f"    checkpoint-like files={len(files)}, size={total_mb:.2f} MB")
    found_any = found_any or bool(files)
    pointer = directory / "last_best_checkpoint"
    if pointer.is_file():
        try:
            name = pointer.read_text(encoding="utf-8").splitlines()[0].strip()
            print(
                f"    last_best_checkpoint -> {name} "
                f"(exists={(directory / name).is_file()})"
            )
        except (OSError, IndexError) as exc:
            print(f"    last_best_checkpoint unreadable: {exc}")

print("\nFRCRN checkpoint found:", found_any)
if not found_any:
    print(
        "No FRCRN checkpoint is present. ClearerVoice would need to download one; "
        "that download is a separate, explicit decision. Do not start inference yet."
    )


=== CHECKPOINT RECOVERY ===
Dataset directory exists: True
Dataset metadata exists: True
STFT checkpoint: /content/drive/MyDrive/Intelli-Speech-data/analysis_cache/stft_example.npz
  sample_rate=16000 n_fft=512 hop_length=128
  clean_stft=(257, 510) noisy_stft=(257, 510)
  all values finite=True

FRCRN checkpoint locations (read-only):
  CWD checkpoints/FRCRN_SE_16K: missing (/content/checkpoints/FRCRN_SE_16K)
  repo checkpoints/FRCRN_SE_16K: missing (/content/Intelli-Speech/checkpoints/FRCRN_SE_16K)
  /content checkpoints/FRCRN_SE_16K: missing (/content/checkpoints/FRCRN_SE_16K)
  persistent_cache/clearvoice_runtime: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/clearvoice_runtime
    checkpoint-like files=0, size=0.00 MB
  persistent_cache/huggingface/hub: /content/drive/MyDrive/Intelli-Speech-data/persistent_cache/huggingface/hub
    checkpoint-like files=0, size=0.00 MB
  user ClearerVoice cache: missing (/root/.cache/clearvoice)
  user Hugging Face cache: missing (/r

In [12]:
# ============================================================
# STEP 6 | CLEARERVOICE RECOVERY (inspection only)
# Checks importability and declared constraints. Installs nothing.
# ============================================================
import importlib.metadata
import importlib.util
import inspect
import platform


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print("=== CLEARERVOICE RECOVERY ===")
print("Python:", platform.python_version())
print("NumPy:", package_version("numpy"))
print("ClearerVoice:", package_version("clearvoice"))

spec = importlib.util.find_spec("clearvoice")
print("ClearerVoice importable:", spec is not None)

if spec is not None:
    try:
        requirements = importlib.metadata.requires("clearvoice") or []
    except importlib.metadata.PackageNotFoundError:
        requirements = []
    relevant = [
        item
        for item in requirements
        if item.lower().startswith(("numpy", "torch", "torchaudio", "opencv", "librosa"))
    ]
    for item in relevant:
        print(f"  declared requirement: {item}")

    try:
        from clearvoice import ClearVoice

        print("Import succeeded:", ClearVoice)
        print("ClearVoice.__call__ signature:", inspect.signature(ClearVoice.__call__))
        print(
            "Model instantiation is intentionally skipped: it can trigger a "
            "checkpoint download. Instantiate only in an approved experiment cell."
        )
    except Exception as exc:
        print(f"Import failed: {type(exc).__name__}: {exc}")
else:
    print(
        "ClearerVoice is not installed. The recorded failure mode was a NumPy "
        "downgrade to 1.26.4 followed by ModuleNotFoundError, so a plain "
        "installation of the package is not a safe recovery path in this runtime."
    )

print("\nManual recovery options, for an explicit decision only (not executed):")
print("  A) Isolated environment (recommended): create a separate virtualenv or a")
print("     dedicated Colab runtime pinned to numpy<2.0 and clearvoice==0.1.2,")
print("     then run the FRCRN experiment there.")
print("  B) Pinned runtime: install the exact set from requirements-colab.txt in a")
print("     fresh runtime, accepting numpy<2.0 for the whole session.")
print("  C) Do nothing yet: keep the current NumPy and continue baseline work that")
print("     does not need ClearerVoice.")
print("\nWhichever option is chosen, record the decision in PROJECT_NOTES.md first.")
print("No package installation, upgrade, or downgrade was performed.")


=== CLEARERVOICE RECOVERY ===
Python: 3.13.18
NumPy: 2.5.5
ClearerVoice: 0.1.2
ClearerVoice importable: True
  declared requirement: librosa==0.10.2.post1
  declared requirement: numpy<2.0,>=1.24.3
  declared requirement: opencv-python==4.10.0.84
  declared requirement: torch>=2.0.1
  declared requirement: torchaudio>=2.0.2
  declared requirement: torchinfo
  declared requirement: torchvision
Import succeeded: <class 'clearvoice.ClearVoice'>
ClearVoice.__call__ signature: (self, input_path, online_write=False, output_path=None)
Model instantiation is intentionally skipped: it can trigger a checkpoint download. Instantiate only in an approved experiment cell.

Manual recovery options, for an explicit decision only (not executed):
  A) Isolated environment (recommended): create a separate virtualenv or a
     dedicated Colab runtime pinned to numpy<2.0 and clearvoice==0.1.2,
     then run the FRCRN experiment there.
  B) Pinned runtime: install the exact set from requirements-colab.txt i

## Recovery checklist

A runtime is recovered when all of the following are true:

- [ ] Step 1 reports the expected Python, NumPy, PyTorch, and torchaudio versions.
- [ ] Step 2 shows Drive mounted and the repository at a known commit.
- [ ] Step 3 created the four cache directories under `persistent_cache/`.
- [ ] Step 4 shows the Hugging Face token state (present or intentionally absent).
- [ ] Step 5 found the STFT checkpoint and reported whether an FRCRN checkpoint exists.
- [ ] Step 6 reported the ClearerVoice state without changing any package.

Then open `MAINFILE.ipynb` and run it top to bottom. `MAINFILE.ipynb` performs no setup
and no recovery; if it fails its precondition check, return here.

## Not covered here

- Installing or downgrading packages.
- Downloading checkpoints or datasets.
- FRCRN inference and experiment output writing.

These require an explicit decision recorded in `PROJECT_NOTES.md`.
